# Visión en LLMs: Análisis de Imágenes con Claude

> Aprende a usar las capacidades de visión de Claude para analizar imágenes desde URL,
> extraer datos estructurados y comparar imágenes.

## Objetivos
- Analizar imágenes desde URL con Claude Vision
- Extraer datos estructurados de imágenes (facturas, documentos)
- Comparar dos imágenes y encontrar diferencias
- Implementar una función `describe_image` reutilizable

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic requests pillow --quiet

## 2. Configuración inicial

In [ ]:
import anthropic
import base64
import json
import requests
from IPython.display import Image, display

client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

def url_a_base64(url: str) -> tuple[str, str]:
    """Descarga una imagen desde URL y la convierte a base64."""
    respuesta = requests.get(url, timeout=10)
    respuesta.raise_for_status()
    content_type = respuesta.headers.get("Content-Type", "image/jpeg").split(";")[0]
    imagen_b64 = base64.b64encode(respuesta.content).decode("utf-8")
    return imagen_b64, content_type

print(f"Cliente inicializado. Modelo: {MODELO}")

## 3. Función `describe_image` reutilizable

Implementamos una función genérica para analizar cualquier imagen con una instrucción personalizada.

In [ ]:
def describe_image(url: str, instruccion: str = "Describe esta imagen en detalle en español.",
                   max_tokens: int = 512) -> str:
    """Analiza una imagen desde URL usando Claude Vision.

    Args:
        url: URL pública de la imagen
        instruccion: Qué queremos que haga Claude con la imagen
        max_tokens: Límite de tokens en la respuesta

    Returns:
        Texto con el análisis de la imagen
    """
    imagen_b64, media_type = url_a_base64(url)

    mensaje = client.messages.create(
        model=MODELO,
        max_tokens=max_tokens,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": media_type,
                            "data": imagen_b64
                        }
                    },
                    {
                        "type": "text",
                        "text": instruccion
                    }
                ]
            }
        ]
    )
    return mensaje.content[0].text

# Prueba rápida con una imagen de ejemplo (diagrama simple)
URL_EJEMPLO = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Git_merge_-_basic_case.svg/400px-Git_merge_-_basic_case.svg.png"
print("Analizando imagen de ejemplo...")
display(Image(url=URL_EJEMPLO, width=400))
descripcion = describe_image(URL_EJEMPLO, "Describe brevemente qué representa este diagrama.")
print(f"\nDescripción:\n{descripcion}")

## 4. Extraer datos estructurados de una imagen de factura

Usamos Claude para extraer campos específicos en formato JSON desde una imagen de factura.

In [ ]:
# Imagen de factura de prueba (factura de muestra pública)
URL_FACTURA = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg"

# Instrucción para extracción estructurada
INSTRUCCION_EXTRACCION = """
Analiza esta imagen y describe lo que ves en formato JSON estructurado.
Incluye los siguientes campos:
{
  "tipo_imagen": "<qué tipo de imagen es>",
  "descripcion_principal": "<descripción del sujeto principal>",
  "colores_dominantes": ["<lista de colores>"],
  "elementos_detectados": ["<lista de elementos visibles>"],
  "calidad_imagen": "<alta/media/baja>",
  "contexto": "<descripción del contexto o fondo>"
}
Responde ÚNICAMENTE con el JSON, sin texto adicional.
"""

print("Extrayendo datos estructurados de la imagen...")
display(Image(url=URL_FACTURA, width=300))

resultado_raw = describe_image(URL_FACTURA, INSTRUCCION_EXTRACCION, max_tokens=512)

# Parsear el JSON
try:
    texto = resultado_raw.strip()
    if texto.startswith("```"):
        texto = texto.split("```")[1]
        if texto.startswith("json"):
            texto = texto[4:]
    datos = json.loads(texto)
    print("\nDatos extraídos:")
    print(json.dumps(datos, indent=2, ensure_ascii=False))
except json.JSONDecodeError:
    print("Respuesta raw:")
    print(resultado_raw)

## 5. Comparar dos imágenes

Claude puede analizar múltiples imágenes en el mismo mensaje y compararlas.

In [ ]:
def comparar_imagenes(url1: str, url2: str, aspecto: str = "visual") -> str:
    """Compara dos imágenes usando Claude Vision.

    Args:
        url1: URL de la primera imagen
        url2: URL de la segunda imagen
        aspecto: Qué aspecto comparar (visual, contenido, estilo, etc.)

    Returns:
        Análisis comparativo en texto
    """
    img1_b64, mt1 = url_a_base64(url1)
    img2_b64, mt2 = url_a_base64(url2)

    mensaje = client.messages.create(
        model=MODELO,
        max_tokens=512,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Imagen 1:"},
                    {"type": "image", "source": {"type": "base64", "media_type": mt1, "data": img1_b64}},
                    {"type": "text", "text": "Imagen 2:"},
                    {"type": "image", "source": {"type": "base64", "media_type": mt2, "data": img2_b64}},
                    {
                        "type": "text",
                        "text": f"Compara estas dos imágenes en términos de {aspecto}. "
                                f"Describe las similitudes y diferencias principales en español."
                    }
                ]
            }
        ]
    )
    return mensaje.content[0].text

# Comparar dos imágenes de insectos diferentes (imágenes públicas de Wikimedia)
URL_IMG1 = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/320px-Camponotus_flavomarginatus_ant.jpg"
URL_IMG2 = "https://upload.wikimedia.org/wikipedia/commons/thumb/7/7e/Diabrotica_speciosa.jpg/320px-Diabrotica_speciosa.jpg"

print("Comparando dos imágenes...")
print("Imagen 1:")
display(Image(url=URL_IMG1, width=250))
print("Imagen 2:")
display(Image(url=URL_IMG2, width=250))

comparacion = comparar_imagenes(URL_IMG1, URL_IMG2, aspecto="características visuales y tipo de insecto")
print(f"\nComparación:\n{comparacion}")

## 6. Demostración completa: describe_image en múltiples modos

In [ ]:
URL_DEMO = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Git_merge_-_basic_case.svg/400px-Git_merge_-_basic_case.svg.png"

modos = [
    ("Descripción general",
     "Describe esta imagen en detalle en español."),
    ("Análisis técnico",
     "¿Qué concepto técnico representa este diagrama? Explícalo paso a paso."),
    ("Texto alternativo (accesibilidad)",
     "Genera un texto alternativo (alt text) conciso para esta imagen, máximo 2 frases."),
]

print("=== Análisis multi-modo de la misma imagen ===")
display(Image(url=URL_DEMO, width=400))
print()

for nombre, instruccion in modos:
    print(f"--- {nombre} ---")
    resultado = describe_image(URL_DEMO, instruccion, max_tokens=256)
    print(resultado)
    print()